In [ ]:
"""
=============================================================================
DCGAN — Synthetic Mammography Multi-View (CC + MLO)
Arsitektur : Deep Convolutional GAN (Radford et al., 2015)
Dataset    : VinDr-Mammo | Kelas: Malignant
Target     : ~6000–7000 pasangan sintetis untuk mengatasi class imbalance
Optimasi   : Mixed Precision (AMP) + T4-friendly
=============================================================================
"""

# ===========================================================================
# 1. IMPORTS
# ===========================================================================

import os, glob, re, time
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# AMP API baru (torch.amp) + fallback utk torch versi lama
try:
    from torch.amp import GradScaler, autocast
except Exception:  # pragma: no cover
    from torch.cuda.amp import GradScaler, autocast

import torchvision.transforms as transforms
import torchvision.utils as vutils


# ===========================================================================
# 2. KONFIGURASI
# ===========================================================================

class Config:
    # ── Path ──────────────────────────────────────────────────────────────
    # NOTE: Path lokal Windows (ganti dari path Kaggle)
    DATA_ROOT   = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\dataset_birads\train"

    # Semua output disimpan ke folder lokal berikut (BI-RADS Sintetis)
    OUTPUT_ROOT = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\BI-RADS Sintetis\synthetic_dataset\train\Malignant"
    MODEL_DIR   = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\BI-RADS Sintetis\models"
    SAMPLE_DIR  = r"B:\Belajar\Skripsi guwa\My Dataset Breast Cancer\BI-RADS Sintetis\samples"

    TARGET_CLASS = "Malignant"

    # ── Resolusi & Training ────────────────────────────────────────────────
    IMG_SIZE    = 128
    EPOCHS      = 400

    # GTX 1050 biasanya VRAM kecil; kalau CUDA sudah aktif tapi OOM, turunkan ini.
    BATCH_SIZE  = 32

    # ── Arsitektur DCGAN ───────────────────────────────────────────────────
    Z_DIM   = 124
    NGF     = 128
    NDF     = 128
    N_CHAN  = 2

    # ── Optimizer ──────────────────────────────────────────────────────────
    LR_G    = 0.0002
    LR_D    = 0.0001
    BETA1   = 0.5
    BETA2   = 0.999

    # ── Mixed Precision ────────────────────────────────────────────────────
    # AMP hanya aktif kalau CUDA tersedia.
    USE_AMP = True

    # ── Generasi sintetis ───────────────────────────────────────────────────
    MALIGNANT_REAL = 767
    BENIGN_COUNT   = 7232
    TARGET_SYNTH   = 6400
    GEN_BATCH_SIZE = 128

    # ── Misc ────────────────────────────────────────────────────────────────
    SEED            = 42

    # Windows sering crash kalau DataLoader pakai multiprocessing.
    NUM_WORKERS     = 0

    SAVE_INTERVAL   = 20
    SAMPLE_INTERVAL = 10

    # ── Label Smoothing ────────────────────────────────────────────────────
    REAL_LABEL_SMOOTH = 0.9
    FAKE_LABEL        = 0.0

    # ── Model selection untuk generasi ────────────────────────────────────
    USE_LAST_EPOCH = True

cfg = Config()

torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device : {device}")
if device.type == "cuda":
    print(f"[INFO] GPU    : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[INFO] VRAM   : {vram:.1f} GB")

# AMP hanya untuk CUDA
amp_enabled = bool(cfg.USE_AMP and device.type == "cuda")
print(f"[INFO] AMP    : {amp_enabled}")

os.makedirs(os.path.join(cfg.OUTPUT_ROOT, "CC"),  exist_ok=True)
os.makedirs(os.path.join(cfg.OUTPUT_ROOT, "MLO"), exist_ok=True)
os.makedirs(cfg.MODEL_DIR, exist_ok=True)
os.makedirs(cfg.SAMPLE_DIR, exist_ok=True)


# ===========================================================================
# 3. DATASET
# ===========================================================================

class MammographyPairDataset(Dataset):
    """
    Memuat pasangan CC + MLO dari satu kelas.
    Output per item: (2, H, W) — grayscale, channel 0=CC, channel 1=MLO
    """
    def __init__(self, root, label_class, img_size, augment=True):
        self.img_size = img_size
        cc_dir  = os.path.join(root, label_class, "CC")
        mlo_dir = os.path.join(root, label_class, "MLO")

        cc_files  = {os.path.basename(f) for f in glob.glob(os.path.join(cc_dir,  "*.png"))}
        mlo_files = {os.path.basename(f) for f in glob.glob(os.path.join(mlo_dir, "*.png"))}
        common    = sorted(cc_files & mlo_files)

        self.pairs = [
            (os.path.join(cc_dir, fn), os.path.join(mlo_dir, fn))
            for fn in common
        ]
        print(f"[Dataset] {label_class} @ {img_size}px: {len(self.pairs)} pasangan")

        aug_list = [
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(degrees=10),
            transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        ] if augment else []

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            *aug_list,
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),   # [-1, 1]
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        cc_path, mlo_path = self.pairs[idx]
        cc  = Image.open(cc_path).convert("L")
        mlo = Image.open(mlo_path).convert("L")
        # Stack → (2, H, W)
        return torch.cat([self.transform(cc), self.transform(mlo)], dim=0)

    def get_max_id(self):
        ids = []
        for p, _ in self.pairs:
            m = re.search(r"breast_(\d+)\.png", os.path.basename(p))
            if m:
                ids.append(int(m.group(1)))
        return max(ids) if ids else 0


# ===========================================================================
# 4. MODEL — DCGAN
# ===========================================================================

def weights_init(m):
    """Inisialisasi bobot sesuai paper DCGAN (mean=0, std=0.02)."""
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ---------------------------------------------------------------------------
# 4a. Generator
#
#   Input : z (B, Z_DIM, 1, 1)
#   Output: (B, N_CHAN, IMG_SIZE, IMG_SIZE)
#
#   Untuk IMG_SIZE=64:
#     z(100,1,1) → 512×4×4 → 256×8×8 → 128×16×16 → 64×32×32 → 2×64×64
#
#   Untuk IMG_SIZE=128, tambah satu DecoderBlock lagi (lihat komentar).
#   Untuk IMG_SIZE=224, gunakan adaptive interpolation di forward().
# ---------------------------------------------------------------------------

class Generator(nn.Module):
    def __init__(self, z_dim: int, ngf: int, n_chan: int, img_size: int = 64):
        super().__init__()
        self.img_size = img_size

        # Hitung berapa blok upsampling yang dibutuhkan
        # Setiap blok: x2 resolusi, mulai dari 4×4
        # 64  → 4 blok (4→8→16→32→64)
        # 128 → 5 blok (4→8→16→32→64→128)
        # 224 → 5 blok + interpolate terakhir ke 224

        n_blocks = max(4, int(np.log2(img_size)) - 1)   # min 4 blok untuk 64px

        layers = [
            # Blok pertama: z(B, Z_DIM, 1, 1) → (B, ngf*8, 4, 4)
            nn.ConvTranspose2d(z_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
        ]

        in_ch  = ngf * 8
        out_ch = ngf * 4
        for i in range(n_blocks - 1):
            layers += [
                nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(True),
            ]
            in_ch  = out_ch
            out_ch = max(out_ch // 2, ngf)   # floor ke ngf

        # Head: channel terakhir → n_chan (CC + MLO)
        layers += [
            nn.ConvTranspose2d(in_ch, n_chan, 4, 2, 1, bias=False),
            nn.Tanh(),
        ]

        self.net = nn.Sequential(*layers)
        self.apply(weights_init)

    def forward(self, z):
        # z: (B, Z_DIM) → (B, Z_DIM, 1, 1)
        x = z.view(z.size(0), -1, 1, 1)
        out = self.net(x)
        # Resize jika resolusi output ≠ img_size (contoh: 224)
        if out.shape[-1] != self.img_size:
            out = nn.functional.interpolate(
                out, size=self.img_size, mode="bilinear", align_corners=False
            )
        return out


# ---------------------------------------------------------------------------
# 4b. Discriminator
#
#   Input : (B, N_CHAN, IMG_SIZE, IMG_SIZE)
#   Output: (B,) — logit skalar per sampel
#
#   Setiap blok Conv2d: stride=2, kernel=4 → resolusi ÷2
#   64×64 → 32→16→8→4 → flatten → linear → 1
# ---------------------------------------------------------------------------

class Discriminator(nn.Module):
    def __init__(self, ndf: int, n_chan: int):
        super().__init__()

        self.net = nn.Sequential(
            # Blok pertama: tidak pakai BN (sesuai paper DCGAN)
            nn.Conv2d(n_chan, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf,     ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # Output: (B, 1, 4, 4) → flatten → (B, 1)
            # Untuk input 64×64: 4 blok stride-2 → 4×4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )
        self.apply(weights_init)

    def forward(self, x):
        # Resize ke 64 agar conv terakhir selalu menghasilkan (B,1,1,1)
        if x.shape[-1] != 64:
            x = nn.functional.interpolate(x, size=64, mode="bilinear", align_corners=False)
        return self.net(x).view(-1)


# ===========================================================================
# 5. TRAINING DCGAN
# ===========================================================================

def train_dcgan(generator: Generator, discriminator: Discriminator):
    dataset = MammographyPairDataset(
        cfg.DATA_ROOT, cfg.TARGET_CLASS, cfg.IMG_SIZE, augment=True
    )

    # Guard: Windows multiprocessing DataLoader sering "worker exited unexpectedly"
    safe_num_workers = int(cfg.NUM_WORKERS)
    if os.name == "nt" and safe_num_workers != 0:
        print("[WARN] Windows detected: override NUM_WORKERS -> 0 (stabil)")
        safe_num_workers = 0

    loader = DataLoader(
        dataset,
        batch_size  = cfg.BATCH_SIZE,
        shuffle     = True,
        num_workers = safe_num_workers,
        pin_memory  = device.type == "cuda",
        drop_last   = True,
    )

    criterion = nn.BCEWithLogitsLoss()

    opt_g = optim.Adam(generator.parameters(),     lr=cfg.LR_G, betas=(cfg.BETA1, cfg.BETA2))
    opt_d = optim.Adam(discriminator.parameters(), lr=cfg.LR_D, betas=(cfg.BETA1, cfg.BETA2))

    # torch.amp.GradScaler(device) pada versi baru
    try:
        scaler_g = GradScaler("cuda", enabled=amp_enabled)
        scaler_d = GradScaler("cuda", enabled=amp_enabled)
    except TypeError:
        scaler_g = GradScaler(enabled=amp_enabled)
        scaler_d = GradScaler(enabled=amp_enabled)

    fixed_noise = torch.randn(16, cfg.Z_DIM, device=device)
    best_g_loss = float("inf")
    best_path   = os.path.join(cfg.MODEL_DIR, "dcgan_generator_best.pth")
    last_path   = os.path.join(cfg.MODEL_DIR, "dcgan_generator_last.pth")

    print(f"\n{'='*60}")
    print(f"  DCGAN Training | {cfg.IMG_SIZE}×{cfg.IMG_SIZE} | {cfg.EPOCHS} epoch | batch {cfg.BATCH_SIZE}")
    print(f"  AMP         : {amp_enabled}")
    print(f"  num_workers : {safe_num_workers}")
    print(f"  Model untuk generasi : {'epoch terakhir' if cfg.USE_LAST_EPOCH else 'best G_loss'}")
    print(f"{'='*60}")

    for epoch in range(1, cfg.EPOCHS + 1):
        generator.train()
        discriminator.train()

        d_loss_sum = g_loss_sum = 0.0
        t0 = time.time()

        for real_imgs in loader:
            real_imgs = real_imgs.to(device)
            B = real_imgs.size(0)

            real_labels = torch.full((B,), cfg.REAL_LABEL_SMOOTH, device=device)
            fake_labels = torch.full((B,), cfg.FAKE_LABEL,        device=device)

            # ── Update Discriminator ───────────────────────────────────
            opt_d.zero_grad()
            with autocast(device_type="cuda", enabled=amp_enabled):
                d_real   = discriminator(real_imgs)
                loss_d_r = criterion(d_real, real_labels)

                # Fake
                z      = torch.randn(B, cfg.Z_DIM, device=device)
                fake   = generator(z).detach()
                d_fake = discriminator(fake)
                loss_d_f = criterion(d_fake, fake_labels)

                loss_d = (loss_d_r + loss_d_f) * 0.5

            scaler_d.scale(loss_d).backward()
            scaler_d.unscale_(opt_d)
            nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
            scaler_d.step(opt_d)
            scaler_d.update()

            # ── Update Generator ───────────────────────────────────────
            opt_g.zero_grad()
            with autocast(device_type="cuda", enabled=amp_enabled):
                z      = torch.randn(B, cfg.Z_DIM, device=device)
                fake   = generator(z)
                d_fake = discriminator(fake)
                # Generator ingin D mengira fake = real
                loss_g = criterion(d_fake, real_labels)

            scaler_g.scale(loss_g).backward()
            scaler_g.unscale_(opt_g)
            nn.utils.clip_grad_norm_(generator.parameters(), 1.0)
            scaler_g.step(opt_g)
            scaler_g.update()

            d_loss_sum += loss_d.item()
            g_loss_sum += loss_g.item()

        n = len(loader)
        avg_d = d_loss_sum / max(n, 1)
        avg_g = g_loss_sum / max(n, 1)
        elapsed = time.time() - t0
        print(f"  Epoch {epoch:>3}/{cfg.EPOCHS}  D:{avg_d:.4f}  G:{avg_g:.4f}  {elapsed:.1f}s")

        # Simpan best model (berdasarkan G_loss terendah)
        if avg_g < best_g_loss:
            best_g_loss = avg_g
            torch.save({
                "generator":     generator.state_dict(),
                "discriminator": discriminator.state_dict(),
                "g_loss":        best_g_loss,
                "epoch":         epoch,
            }, best_path)
            print(f"  ✓ Best model disimpan (G_loss={best_g_loss:.4f}, epoch={epoch})")

        # Simpan sampel gambar
        if epoch % cfg.SAMPLE_INTERVAL == 0:
            generator.eval()
            with torch.no_grad():
                with autocast(device_type="cuda", enabled=amp_enabled):
                    fake_sample = generator(fixed_noise)

            vutils.save_image(
                fake_sample[:, 0:1],
                os.path.join(cfg.SAMPLE_DIR, f"cc_ep{epoch}.png"),
                normalize=True, nrow=4
            )
            vutils.save_image(
                fake_sample[:, 1:2],
                os.path.join(cfg.SAMPLE_DIR, f"mlo_ep{epoch}.png"),
                normalize=True, nrow=4
            )

    # ── Simpan state epoch terakhir ────────────────────────────────────────
    torch.save({
        "generator":     generator.state_dict(),
        "discriminator": discriminator.state_dict(),
        "epoch":         cfg.EPOCHS,
    }, last_path)
    print(f"\n  ✓ Last epoch model disimpan → {last_path}")

    # ── Pilih model untuk generasi ─────────────────────────────────────────
    if cfg.USE_LAST_EPOCH:
        # Generator sudah dalam state epoch terakhir — tidak perlu load ulang
        print(f"[INFO] USE_LAST_EPOCH=True → memakai generator epoch terakhir ({cfg.EPOCHS})")
    else:
        # Load best model berdasarkan G_loss
        ckpt = torch.load(best_path, map_location=device)
        generator.load_state_dict(ckpt["generator"])
        print(f"[INFO] USE_LAST_EPOCH=False → generator terbaik dimuat "
              f"(epoch {ckpt['epoch']}, G_loss={ckpt['g_loss']:.4f})")

    return generator


# ===========================================================================
# 6. GENERASI DATA SINTETIS
# ===========================================================================

def tensor_to_pil(t: torch.Tensor) -> Image.Image:
    """Konversi tensor (1, H, W) dari [-1,1] ke PIL Image grayscale."""
    t = t.squeeze(0).cpu().float()
    t = ((t + 1.0) / 2.0).clamp(0, 1)
    return Image.fromarray((t.numpy() * 255).astype(np.uint8), mode="L")


def generate_synthetic_pairs(generator: Generator, start_id: int, n_pairs: int):
    cc_dir  = os.path.join(cfg.OUTPUT_ROOT, "CC")
    mlo_dir = os.path.join(cfg.OUTPUT_ROOT, "MLO")

    generator.eval()
    generated  = 0
    current_id = start_id
    t0         = time.time()

    mode_label = "epoch terakhir" if cfg.USE_LAST_EPOCH else "best model"
    print(f"\n[INFO] Generating {n_pairs} pasangan sintetis @ {cfg.IMG_SIZE}×{cfg.IMG_SIZE}...")
    print(f"[INFO] Generator  : {mode_label}")
    print(f"[INFO] File ID mulai: breast_{start_id}.png")

    with torch.no_grad():
        while generated < n_pairs:
            this_batch = min(cfg.GEN_BATCH_SIZE, n_pairs - generated)
            z    = torch.randn(this_batch, cfg.Z_DIM, device=device)

            with autocast(enabled=cfg.USE_AMP):
                fake = generator(z)            # (B, 2, H, W)

            cc_batch  = fake[:, 0:1]           # (B, 1, H, W)
            mlo_batch = fake[:, 1:2]           # (B, 1, H, W)

            for i in range(this_batch):
                fname = f"breast_{current_id}.png"
                tensor_to_pil(cc_batch[i]).save(os.path.join(cc_dir,  fname))
                tensor_to_pil(mlo_batch[i]).save(os.path.join(mlo_dir, fname))
                current_id += 1

            generated += this_batch

            if generated % 500 == 0 or generated == n_pairs:
                elapsed = time.time() - t0
                rate    = generated / max(elapsed, 1e-6)
                eta     = (n_pairs - generated) / max(rate, 1e-6)
                print(f"  [{generated:>6}/{n_pairs}]  {rate:.0f} pair/s  ETA: {eta:.0f}s")

    print(f"[INFO] Selesai. ID range: breast_{start_id} → breast_{current_id-1}")


# ===========================================================================
# 7. VERIFIKASI
# ===========================================================================

def verify_output():
    cc_files  = {os.path.basename(f) for f in glob.glob(os.path.join(cfg.OUTPUT_ROOT, "CC",  "*.png"))}
    mlo_files = {os.path.basename(f) for f in glob.glob(os.path.join(cfg.OUTPUT_ROOT, "MLO", "*.png"))}

    missing_mlo = cc_files - mlo_files
    missing_cc  = mlo_files - cc_files

    total_mal = cfg.MALIGNANT_REAL + len(cc_files)
    ratio     = total_mal / cfg.BENIGN_COUNT * 100

    print(f"\n{'='*60}")
    print(f"  VERIFIKASI OUTPUT")
    print(f"{'='*60}")
    print(f"  CC  : {len(cc_files)} file")
    print(f"  MLO : {len(mlo_files)} file")

    if not missing_mlo and not missing_cc:
        print(f"  ✓ Semua pasangan simetris dan lengkap!")
    else:
        if missing_mlo: print(f"  ⚠ CC tanpa pasangan MLO : {len(missing_mlo)}")
        if missing_cc:  print(f"  ⚠ MLO tanpa pasangan CC : {len(missing_cc)}")

    print(f"\n  Distribusi Kelas Akhir:")
    print(f"    Benign    (asli)     : {cfg.BENIGN_COUNT:>6}")
    print(f"    Malignant (asli)     : {cfg.MALIGNANT_REAL:>6}")
    print(f"    Malignant (sintetis) : {len(cc_files):>6}")
    print(f"    Malignant (total)    : {total_mal:>6}")
    print(f"    Class ratio          : {ratio:.1f}% dari Benign")
    print(f"{'='*60}")


# ===========================================================================
# 8. MAIN
# ===========================================================================

def main():
    print("="*60)
    print("  DCGAN — VinDr-Mammo Synthetic Generation")
    print("  Arsitektur: Radford et al. (2015) — tanpa pretrained")
    print(f"  Mode generasi: {'epoch terakhir' if cfg.USE_LAST_EPOCH else 'best G_loss'}")
    print("="*60)

    # Cek max ID dataset asli
    dummy_ds = MammographyPairDataset(cfg.DATA_ROOT, cfg.TARGET_CLASS, 64, augment=False)
    max_id   = dummy_ds.get_max_id()
    start_id = max_id + 1
    del dummy_ds
    print(f"[INFO] Max ID asli: breast_{max_id}.png → sintetis mulai breast_{start_id}.png")

    # Inisialisasi DCGAN
    generator     = Generator(cfg.Z_DIM, cfg.NGF, cfg.N_CHAN, cfg.IMG_SIZE).to(device)
    discriminator = Discriminator(cfg.NDF, cfg.N_CHAN).to(device)

    n_g = sum(p.numel() for p in generator.parameters())
    n_d = sum(p.numel() for p in discriminator.parameters())
    print(f"[INFO] Generator params     : {n_g:,}")
    print(f"[INFO] Discriminator params : {n_d:,}")

    # Training
    generator = train_dcgan(generator, discriminator)

    # Generate data sintetis
    generate_synthetic_pairs(generator, start_id, cfg.TARGET_SYNTH)

    # Verifikasi output
    verify_output()

    print("\n[DONE] Pipeline selesai!")
    print(f"       Sintetis  : {cfg.OUTPUT_ROOT}")
    print(f"       Best model: {cfg.MODEL_DIR}/dcgan_generator_best.pth")
    print(f"       Last model: {cfg.MODEL_DIR}/dcgan_generator_last.pth")
    print(f"       Samples   : {cfg.SAMPLE_DIR}/")


if __name__ == "__main__":
    main()

[INFO] Device : cuda
[INFO] GPU    : Tesla T4
[INFO] VRAM   : 15.6 GB
  DCGAN — VinDr-Mammo Synthetic Generation
  Arsitektur: Radford et al. (2015) — tanpa pretrained
  Mode generasi: epoch terakhir
[Dataset] Malignant @ 64px: 767 pasangan
[INFO] Max ID asli: breast_9991.png → sintetis mulai breast_9992.png
[INFO] Generator params     : 13,574,400
[INFO] Discriminator params : 11,034,112
[Dataset] Malignant @ 128px: 767 pasangan

  DCGAN Training | 128×128 | 400 epoch | batch 64
  Model untuk generasi : epoch terakhir


/tmp/ipykernel_23/3757311414.py:304: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_g = GradScaler(enabled=cfg.USE_AMP)
/tmp/ipykernel_23/3757311414.py:305: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_d = GradScaler(enabled=cfg.USE_AMP)
/tmp/ipykernel_23/3757311414.py:334: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.USE_AMP):
/tmp/ipykernel_23/3757311414.py:355: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.USE_AMP):


  Epoch   1/400  D:1.1463  G:8.2558  25.0s
  ✓ Best model disimpan (G_loss=8.2558, epoch=1)
  Epoch   2/400  D:0.9617  G:13.9086  17.2s
  Epoch   3/400  D:1.6101  G:12.8196  16.2s
  Epoch   4/400  D:1.3718  G:14.5641  16.5s
  Epoch   5/400  D:0.9462  G:15.2703  16.3s
  Epoch   6/400  D:0.9848  G:13.2353  16.5s
  Epoch   7/400  D:1.0495  G:9.6162  16.2s
  Epoch   8/400  D:0.6024  G:9.6636  16.3s
  Epoch   9/400  D:0.7635  G:8.4731  16.2s
  Epoch  10/400  D:0.5199  G:7.5258  16.4s
  ✓ Best model disimpan (G_loss=7.5258, epoch=10)


/tmp/ipykernel_23/3757311414.py:399: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.USE_AMP):


  Epoch  11/400  D:0.6506  G:6.4147  15.9s
  ✓ Best model disimpan (G_loss=6.4147, epoch=11)
  Epoch  12/400  D:0.6345  G:4.9191  16.5s
  ✓ Best model disimpan (G_loss=4.9191, epoch=12)
  Epoch  13/400  D:0.8078  G:3.5639  16.2s
  ✓ Best model disimpan (G_loss=3.5639, epoch=13)
  Epoch  14/400  D:1.0866  G:3.7469  16.1s
  Epoch  15/400  D:0.8157  G:2.7421  16.3s
  ✓ Best model disimpan (G_loss=2.7421, epoch=15)
  Epoch  16/400  D:0.8478  G:2.8895  16.3s
  Epoch  17/400  D:0.6929  G:3.2021  16.4s
  Epoch  18/400  D:0.7096  G:3.2943  16.2s
  Epoch  19/400  D:0.6634  G:3.0780  16.4s
  Epoch  20/400  D:0.5835  G:3.7384  16.3s
  Epoch  21/400  D:0.6142  G:3.1315  16.5s
  Epoch  22/400  D:0.7334  G:2.4617  16.0s
  ✓ Best model disimpan (G_loss=2.4617, epoch=22)
  Epoch  23/400  D:0.6380  G:2.4359  16.1s
  ✓ Best model disimpan (G_loss=2.4359, epoch=23)
  Epoch  24/400  D:0.6688  G:2.3730  16.1s
  ✓ Best model disimpan (G_loss=2.3730, epoch=24)
  Epoch  25/400  D:0.7353  G:1.7622  16.4s
  ✓ B

/tmp/ipykernel_23/3757311414.py:464: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=cfg.USE_AMP):
/tmp/ipykernel_23/3757311414.py:442: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray((t.numpy() * 255).astype(np.uint8), mode="L")


  [  6400/6400]  354 pair/s  ETA: 0s
[INFO] Selesai. ID range: breast_9992 → breast_16391

  VERIFIKASI OUTPUT
  CC  : 6400 file
  MLO : 6400 file
  ✓ Semua pasangan simetris dan lengkap!

  Distribusi Kelas Akhir:
    Benign    (asli)     :   7232
    Malignant (asli)     :    767
    Malignant (sintetis) :   6400
    Malignant (total)    :   7167
    Class ratio          : 99.1% dari Benign

[DONE] Pipeline selesai!
       Sintetis  : /kaggle/working/synthetic_dataset/train/Malignant
       Best model: /kaggle/working/models/dcgan_generator_best.pth
       Last model: /kaggle/working/models/dcgan_generator_last.pth
       Samples   : /kaggle/working/samples/
